In [5]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

TensorFlow version: 2.21.0
NumPy version: 2.4.6
Pandas version: 3.0.5


In [6]:
# ============================================================
# CELL 2 — LOAD DATASETS
# ============================================================

FEATURE_FILE = "inflation_feature_engineered.csv"
MODELING_FILE = "inflation_modeling_features.csv"
CLUSTER_FILE = "final_country_clusters.csv"

feature_df = pd.read_csv(FEATURE_FILE)
modeling_df = pd.read_csv(MODELING_FILE)
cluster_df = pd.read_csv(CLUSTER_FILE)

print("=" * 60)
print("DATASETS LOADED")
print("=" * 60)

print("\nFeature engineered:")
print(feature_df.shape)

print("\nModeling features:")
print(modeling_df.shape)

print("\nClustering:")
print(cluster_df.shape)

DATASETS LOADED

Feature engineered:
(193, 151)

Modeling features:
(193, 152)

Clustering:
(193, 3)


In [7]:
# ============================================================
# CELL 3 — INSPECT DATASETS
# ============================================================

print("=" * 60)
print("FEATURE ENGINEERED COLUMNS")
print("=" * 60)

print(feature_df.columns.tolist())

print("\n" + "=" * 60)
print("MODELING FEATURE COLUMNS")
print("=" * 60)

print(modeling_df.columns.tolist())

print("\n" + "=" * 60)
print("CLUSTER COLUMNS")
print("=" * 60)

print(cluster_df.columns.tolist())

FEATURE ENGINEERED COLUMNS
['Country Name', 'Country Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Mean_Inflation', 'Median_Inflation', 'Std_Inflation', 'Min_Inflation', 'Max_Inflation', 'Inflation_Range', 'Mean_Recent_5Y', 'Mean_Recent_10Y', 'Mean_Previous_10Y', 'Long_Term_Change', 'Long_Term_Trend', 'Inflation_Volatility', 'High_Inflation_Years', 'Very_High_Inflation_Years', 'Negative_Inflation_Years', 'Diff_1961', 'Diff_1962', 'Diff_1963', 'Diff_1964', 'Diff_1965', 'Diff_1966', 'Diff_1967', 'Diff_1968'

In [8]:
# ============================================================
# CELL 4 — COUNTRY ALIGNMENT CHECK
# ============================================================

print("=" * 60)
print("COUNTRY ALIGNMENT")
print("=" * 60)

print("Feature dataset countries :", len(feature_df))
print("Modeling dataset countries:", len(modeling_df))
print("Cluster dataset countries:", len(cluster_df))

feature_codes = feature_df["Country Code"].astype(str).tolist()
modeling_codes = modeling_df["Country Code"].astype(str).tolist()
cluster_codes = cluster_df["Country Code"].astype(str).tolist()

print("\nFeature vs Modeling:",
      feature_codes == modeling_codes)

print("Feature vs Clustering:",
      feature_codes == cluster_codes)

print("\nNumber of unique countries:")
print("Feature   :", feature_df["Country Code"].nunique())
print("Modeling  :", modeling_df["Country Code"].nunique())
print("Clustering:", cluster_df["Country Code"].nunique())

COUNTRY ALIGNMENT
Feature dataset countries : 193
Modeling dataset countries: 193
Cluster dataset countries: 193

Feature vs Modeling: True
Feature vs Clustering: True

Number of unique countries:
Feature   : 193
Modeling  : 193
Clustering: 193


In [9]:
# ============================================================
# CELL 5 — IDENTIFY YEAR COLUMNS
# ============================================================

YEAR_COLUMNS = [
    str(year)
    for year in range(1960, 2026)
    if str(year) in feature_df.columns
]

print("Number of year columns:", len(YEAR_COLUMNS))

print("\nFirst 10 years:")
print(YEAR_COLUMNS[:10])

print("\nLast 10 years:")
print(YEAR_COLUMNS[-10:])

Number of year columns: 66

First 10 years:
['1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969']

Last 10 years:
['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [10]:
# ============================================================
# CELL 6 — INFLATION MATRIX
# ============================================================

inflation_matrix = feature_df[YEAR_COLUMNS].astype(float).values

country_names = feature_df["Country Name"].values
country_codes = feature_df["Country Code"].values

years = np.array([int(y) for y in YEAR_COLUMNS])

print("=" * 60)
print("INFLATION MATRIX")
print("=" * 60)

print("Shape:", inflation_matrix.shape)
print("Countries:", len(country_codes))
print("Years:", len(years))

print("Missing values:", np.isnan(inflation_matrix).sum())

INFLATION MATRIX
Shape: (193, 66)
Countries: 193
Years: 66
Missing values: 0


In [11]:
# ============================================================
# CELL 7 — CORRECT STATIC FEATURES
# ============================================================

YEAR_COLUMNS = [
    str(year)
    for year in range(1960, 2026)
]

# Columns that must NOT be used as static features
exclude_columns = (
    ["Country Name", "Country Code"]
    + YEAR_COLUMNS
    + ["Cluster"]
)

# Select only the 83 engineered/static features
static_columns = [
    col
    for col in modeling_df.columns
    if col not in exclude_columns
]

static_features = modeling_df[static_columns].copy()

print("=" * 60)
print("STATIC FEATURES — CORRECTED")
print("=" * 60)

print("Number of static features:", len(static_columns))
print("Static feature shape:", static_features.shape)

print("\nStatic feature columns:")
print(static_columns)

print("\nMissing values:",
      static_features.isna().sum().sum())

STATIC FEATURES — CORRECTED
Number of static features: 83
Static feature shape: (193, 83)

Static feature columns:
['Mean_Inflation', 'Median_Inflation', 'Std_Inflation', 'Min_Inflation', 'Max_Inflation', 'Inflation_Range', 'Mean_Recent_5Y', 'Mean_Recent_10Y', 'Mean_Previous_10Y', 'Long_Term_Change', 'Long_Term_Trend', 'Inflation_Volatility', 'High_Inflation_Years', 'Very_High_Inflation_Years', 'Negative_Inflation_Years', 'Diff_1961', 'Diff_1962', 'Diff_1963', 'Diff_1964', 'Diff_1965', 'Diff_1966', 'Diff_1967', 'Diff_1968', 'Diff_1969', 'Diff_1970', 'Diff_1971', 'Diff_1972', 'Diff_1973', 'Diff_1974', 'Diff_1975', 'Diff_1976', 'Diff_1977', 'Diff_1978', 'Diff_1979', 'Diff_1980', 'Diff_1981', 'Diff_1982', 'Diff_1983', 'Diff_1984', 'Diff_1985', 'Diff_1986', 'Diff_1987', 'Diff_1988', 'Diff_1989', 'Diff_1990', 'Diff_1991', 'Diff_1992', 'Diff_1993', 'Diff_1994', 'Diff_1995', 'Diff_1996', 'Diff_1997', 'Diff_1998', 'Diff_1999', 'Diff_2000', 'Diff_2001', 'Diff_2002', 'Diff_2003', 'Diff_2004', 'D

In [12]:
# ============================================================
# CELL 8 — STATIC FEATURE MATRIX
# ============================================================

X_static_base = static_features.astype(float).values

print("=" * 60)
print("STATIC FEATURE MATRIX")
print("=" * 60)

print("Shape:", X_static_base.shape)
print("Missing values:", np.isnan(X_static_base).sum())

STATIC FEATURE MATRIX
Shape: (193, 83)
Missing values: 0


In [13]:
# ============================================================
# CELL 9 — CREATE 10-YEAR SEQUENCES
# ============================================================

SEQUENCE_LENGTH = 10

X_sequences = []
X_static = []
y_values = []

sequence_clusters = []
target_years = []
sequence_country_codes = []

for country_idx in range(len(country_codes)):

    country_series = inflation_matrix[country_idx]

    for i in range(SEQUENCE_LENGTH, len(years)):

        # Previous 10 years
        sequence = country_series[
            i - SEQUENCE_LENGTH:i
        ]

        # Next year's inflation
        target = country_series[i]

        target_year = years[i]

        X_sequences.append(sequence.reshape(SEQUENCE_LENGTH, 1))

        X_static.append(X_static_base[country_idx])

        y_values.append(target)

        target_years.append(target_year)

        sequence_country_codes.append(
            country_codes[country_idx]
        )

X_sequences = np.array(X_sequences)
X_static = np.array(X_static)
y_values = np.array(y_values)
target_years = np.array(target_years)
sequence_country_codes = np.array(sequence_country_codes)

print("=" * 60)
print("SEQUENCE DATA")
print("=" * 60)

print("X_sequences:", X_sequences.shape)
print("X_static   :", X_static.shape)
print("y_values   :", y_values.shape)
print("Target years:", target_years.shape)
print("Countries:", sequence_country_codes.shape)

SEQUENCE DATA
X_sequences: (10808, 10, 1)
X_static   : (10808, 83)
y_values   : (10808,)
Target years: (10808,)
Countries: (10808,)


In [14]:
# ============================================================
# CELL 10 — TRAIN / VALIDATION / TEST SPLIT
# ============================================================

TRAIN_END_YEAR = 2019
VALIDATION_END_YEAR = 2022

train_mask = target_years <= TRAIN_END_YEAR

validation_mask = (
    (target_years > TRAIN_END_YEAR) &
    (target_years <= VALIDATION_END_YEAR)
)

test_mask = target_years > VALIDATION_END_YEAR

X_seq_train = X_sequences[train_mask]
X_seq_val = X_sequences[validation_mask]
X_seq_test = X_sequences[test_mask]

X_static_train = X_static[train_mask]
X_static_val = X_static[validation_mask]
X_static_test = X_static[test_mask]

y_train = y_values[train_mask]
y_val = y_values[validation_mask]
y_test = y_values[test_mask]

print("=" * 60)
print("TRAIN")
print("=" * 60)

print(X_seq_train.shape)
print(X_static_train.shape)
print(y_train.shape)

print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)

print(X_seq_val.shape)
print(X_static_val.shape)
print(y_val.shape)

print("\n" + "=" * 60)
print("TEST")
print("=" * 60)

print(X_seq_test.shape)
print(X_static_test.shape)
print(y_test.shape)

print("\nYears:")
print(
    "Train:",
    target_years[train_mask].min(),
    "-",
    target_years[train_mask].max()
)

print(
    "Validation:",
    target_years[validation_mask].min(),
    "-",
    target_years[validation_mask].max()
)

print(
    "Test:",
    target_years[test_mask].min(),
    "-",
    target_years[test_mask].max()
)

TRAIN
(9650, 10, 1)
(9650, 83)
(9650,)

VALIDATION
(579, 10, 1)
(579, 83)
(579,)

TEST
(579, 10, 1)
(579, 83)
(579,)

Years:
Train: 1970 - 2019
Validation: 2020 - 2022
Test: 2023 - 2025


In [15]:
# ============================================================
# CELL 11 — SEQUENCE SCALING
# ============================================================

sequence_scaler = StandardScaler()

# Flatten training sequences
train_flat = X_seq_train.reshape(-1, 1)

# Fit ONLY on training data
sequence_scaler.fit(train_flat)

# Transform
X_seq_train_scaled = sequence_scaler.transform(
    X_seq_train.reshape(-1, 1)
).reshape(X_seq_train.shape)

X_seq_val_scaled = sequence_scaler.transform(
    X_seq_val.reshape(-1, 1)
).reshape(X_seq_val.shape)

X_seq_test_scaled = sequence_scaler.transform(
    X_seq_test.reshape(-1, 1)
).reshape(X_seq_test.shape)

print("=" * 60)
print("SEQUENCE SCALING")
print("=" * 60)

print("Train:", X_seq_train_scaled.shape)
print("Validation:", X_seq_val_scaled.shape)
print("Test:", X_seq_test_scaled.shape)

print(
    "Training mean:",
    X_seq_train_scaled.mean()
)

print(
    "Training std:",
    X_seq_train_scaled.std()
)

SEQUENCE SCALING
Train: (9650, 10, 1)
Validation: (579, 10, 1)
Test: (579, 10, 1)
Training mean: 2.356203890603441e-18
Training std: 1.0


In [16]:
# ============================================================
# CELL 12 — STATIC FEATURE SCALING
# ============================================================

static_scaler = StandardScaler()

static_scaler.fit(X_static_train)

X_static_train_scaled = static_scaler.transform(
    X_static_train
)

X_static_val_scaled = static_scaler.transform(
    X_static_val
)

X_static_test_scaled = static_scaler.transform(
    X_static_test
)

print("=" * 60)
print("STATIC SCALING")
print("=" * 60)

print("Train:", X_static_train_scaled.shape)
print("Validation:", X_static_val_scaled.shape)
print("Test:", X_static_test_scaled.shape)

print(
    "Training mean:",
    X_static_train_scaled.mean()
)

print(
    "Training std:",
    X_static_train_scaled.std()
)

STATIC SCALING
Train: (9650, 83)
Validation: (579, 83)
Test: (579, 83)
Training mean: -2.316460692448684e-16
Training std: 0.9999999999999996


In [17]:
# ============================================================
# CELL 13 — ROBUST TARGET TRANSFORMATION
# ============================================================

def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


# Transform raw targets
y_train_transformed = signed_log1p(y_train)
y_val_transformed = signed_log1p(y_val)
y_test_transformed = signed_log1p(y_test)

print("=" * 60)
print("ROBUST TARGET TRANSFORMATION")
print("=" * 60)

print("Original training range:")
print(
    y_train.min(),
    "to",
    y_train.max()
)

print("\nTransformed training range:")
print(
    y_train_transformed.min(),
    "to",
    y_train_transformed.max()
)

ROBUST TARGET TRANSFORMATION
Original training range:
-17.6404240920371 to 23773.1317741016

Transformed training range:
-2.9253325607173575 to 10.076353368307306


In [18]:
# ============================================================
# CELL 14 — TARGET STANDARDIZATION
# ============================================================

target_scaler = StandardScaler()

# Fit ONLY on transformed training targets
target_scaler.fit(
    y_train_transformed.reshape(-1, 1)
)

y_train_scaled = target_scaler.transform(
    y_train_transformed.reshape(-1, 1)
).flatten()

y_val_scaled = target_scaler.transform(
    y_val_transformed.reshape(-1, 1)
).flatten()

y_test_scaled = target_scaler.transform(
    y_test_transformed.reshape(-1, 1)
).flatten()

print("=" * 60)
print("TARGET STANDARDIZATION")
print("=" * 60)

print("Train mean:", y_train_scaled.mean())
print("Train std :", y_train_scaled.std())

print("\nMissing values:")
print("Train:", np.isnan(y_train_scaled).sum())
print("Val  :", np.isnan(y_val_scaled).sum())
print("Test :", np.isnan(y_test_scaled).sum())

TARGET STANDARDIZATION
Train mean: -1.133923122352906e-16
Train std : 0.9999999999999999

Missing values:
Train: 0
Val  : 0
Test : 0


In [19]:
# ============================================================
# CELL 15 — FINAL PREPROCESSING CHECK
# ============================================================

print("=" * 60)
print("FINAL PREPROCESSING CHECK")
print("=" * 60)

print("\nTRAIN")
print("Sequence :", X_seq_train_scaled.shape)
print("Static   :", X_static_train_scaled.shape)
print("Target   :", y_train_scaled.shape)

print("\nVALIDATION")
print("Sequence :", X_seq_val_scaled.shape)
print("Static   :", X_static_val_scaled.shape)
print("Target   :", y_val_scaled.shape)

print("\nTEST")
print("Sequence :", X_seq_test_scaled.shape)
print("Static   :", X_static_test_scaled.shape)
print("Target   :", y_test_scaled.shape)

print("\nMissing values:")
print(
    "Sequence:",
    np.isnan(X_seq_train_scaled).sum()
    + np.isnan(X_seq_val_scaled).sum()
    + np.isnan(X_seq_test_scaled).sum()
)

print(
    "Static:",
    np.isnan(X_static_train_scaled).sum()
    + np.isnan(X_static_val_scaled).sum()
    + np.isnan(X_static_test_scaled).sum()
)

print(
    "Target:",
    np.isnan(y_train_scaled).sum()
    + np.isnan(y_val_scaled).sum()
    + np.isnan(y_test_scaled).sum()
)

FINAL PREPROCESSING CHECK

TRAIN
Sequence : (9650, 10, 1)
Static   : (9650, 83)
Target   : (9650,)

VALIDATION
Sequence : (579, 10, 1)
Static   : (579, 83)
Target   : (579,)

TEST
Sequence : (579, 10, 1)
Static   : (579, 83)
Target   : (579,)

Missing values:
Sequence: 0
Static: 0
Target: 0


In [20]:
# ============================================================
# CELL 16 — LOAD SAVED GRU + ATTENTION MODEL
# ============================================================

import tensorflow as tf

MODEL_FILE = "best_gru_attention_model.keras"

model = tf.keras.models.load_model(MODEL_FILE)

print("=" * 60)
print("SAVED MODEL LOADED SUCCESSFULLY")
print("=" * 60)

print("\nModel name:", model.name)
print("Number of layers:", len(model.layers))

print("\nModel inputs:")
for inp in model.inputs:
    print(inp)

print("\nModel output:")
print(model.output)

SAVED MODEL LOADED SUCCESSFULLY

Model name: Shared_GRU_MultiHead_Temporal_Attention
Number of layers: 19

Model inputs:
<KerasTensor shape=(None, 10, 1), dtype=float32, sparse=False, ragged=False, name=sequence_input>
<KerasTensor shape=(None, 83), dtype=float32, sparse=False, ragged=False, name=static_input>

Model output:
<KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_37>


In [21]:
# ============================================================
# CELL 17 — MODEL INPUT COMPATIBILITY CHECK
# ============================================================

print("=" * 60)
print("MODEL INPUT COMPATIBILITY")
print("=" * 60)

print("\nExpected sequence input:")
print(X_seq_test_scaled.shape)

print("\nExpected static input:")
print(X_static_test_scaled.shape)

print("\nModel inputs:")

for i, inp in enumerate(model.inputs):
    print(f"Input {i}: {inp.shape}")

print("\nModel output:")
print(model.output.shape)

MODEL INPUT COMPATIBILITY

Expected sequence input:
(579, 10, 1)

Expected static input:
(579, 83)

Model inputs:
Input 0: (None, 10, 1)
Input 1: (None, 83)

Model output:
(None, 1)


In [22]:
# ============================================================
# DIAGNOSTIC — STATIC FEATURE MISMATCH
# ============================================================

print("=" * 60)
print("STATIC FEATURE MISMATCH CHECK")
print("=" * 60)

print("\nModel expects:")
print(model.inputs[1].shape)

print("\nCurrent static data:")
print("X_static_train:", X_static_train.shape)
print("X_static_val  :", X_static_val.shape)
print("X_static_test :", X_static_test.shape)

print("\nNumber of columns in modeling_df:")
print(len(modeling_df.columns))

print("\nModeling dataframe columns:")
print(modeling_df.columns.tolist())

STATIC FEATURE MISMATCH CHECK

Model expects:
(None, 83)

Current static data:
X_static_train: (9650, 83)
X_static_val  : (579, 83)
X_static_test : (579, 83)

Number of columns in modeling_df:
152

Modeling dataframe columns:
['Country Name', 'Country Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Mean_Inflation', 'Median_Inflation', 'Std_Inflation', 'Min_Inflation', 'Max_Inflation', 'Inflation_Range', 'Mean_Recent_5Y', 'Mean_Recent_10Y', 'Mean_Previous_10Y', 'Long_Term_Change', 'Long_Term_Trend', 'Inflati

In [23]:
# ============================================================
# CHECK FEATURE ENGINEERED DATASET
# ============================================================

print("=" * 60)
print("FEATURE ENGINEERED DATASET")
print("=" * 60)

print("Shape:", feature_df.shape)

print("\nColumns:")
print(feature_df.columns.tolist())

FEATURE ENGINEERED DATASET
Shape: (193, 151)

Columns:
['Country Name', 'Country Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Mean_Inflation', 'Median_Inflation', 'Std_Inflation', 'Min_Inflation', 'Max_Inflation', 'Inflation_Range', 'Mean_Recent_5Y', 'Mean_Recent_10Y', 'Mean_Previous_10Y', 'Long_Term_Change', 'Long_Term_Trend', 'Inflation_Volatility', 'High_Inflation_Years', 'Very_High_Inflation_Years', 'Negative_Inflation_Years', 'Diff_1961', 'Diff_1962', 'Diff_1963', 'Diff_1964', 'Diff_1965', 'Diff_196

In [24]:
# ============================================================
# FINAL INPUT CHECK BEFORE PREDICTION
# ============================================================

print("=" * 60)
print("FINAL MODEL INPUT CHECK")
print("=" * 60)

print("\nMODEL EXPECTS:")

for i, inp in enumerate(model.inputs):
    print(f"Input {i}: {inp.shape}")

print("\nWE ARE PROVIDING:")

print("Sequence:", X_seq_test_scaled.shape)
print("Static  :", X_static_test_scaled.shape)

print("\nExpected:")
print("Sequence: (579, 10, 1)")
print("Static  : (579, 83)")

FINAL MODEL INPUT CHECK

MODEL EXPECTS:
Input 0: (None, 10, 1)
Input 1: (None, 83)

WE ARE PROVIDING:
Sequence: (579, 10, 1)
Static  : (579, 83)

Expected:
Sequence: (579, 10, 1)
Static  : (579, 83)


In [25]:
y_pred_test_scaled = model.predict(
    [
        X_seq_test_scaled,
        X_static_test_scaled
    ],
    verbose=1
).flatten()

print("Scaled prediction shape:", y_pred_test_scaled.shape)
print("Scaled actual shape:", y_test_scaled.shape)

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step
Scaled prediction shape: (579,)
Scaled actual shape: (579,)


In [26]:
# ============================================================
# CELL 19 — INVERSE TRANSFORM PREDICTIONS
# ============================================================

def inverse_signed_log1p(x):
    return np.sign(x) * np.expm1(np.abs(x))


# ------------------------------------------------------------
# Step 1: Reverse target StandardScaler
# ------------------------------------------------------------

y_pred_transformed = target_scaler.inverse_transform(
    y_pred_test_scaled.reshape(-1, 1)
).flatten()

y_actual_transformed = target_scaler.inverse_transform(
    y_test_scaled.reshape(-1, 1)
).flatten()


# ------------------------------------------------------------
# Step 2: Reverse signed-log transformation
# ------------------------------------------------------------

y_pred_test = inverse_signed_log1p(
    y_pred_transformed
)

y_actual_test = inverse_signed_log1p(
    y_actual_transformed
)


print("=" * 60)
print("PREDICTIONS CONVERTED TO ORIGINAL SCALE")
print("=" * 60)

print("\nActual minimum    :", y_actual_test.min())
print("Actual maximum    :", y_actual_test.max())

print("\nPredicted minimum :", y_pred_test.min())
print("Predicted maximum :", y_pred_test.max())

PREDICTIONS CONVERTED TO ORIGINAL SCALE

Actual minimum    : -6.601185640737259
Actual maximum    : 254.9485347818159

Predicted minimum : 0.06414961
Predicted maximum : 204.53197


In [27]:
# ============================================================
# CELL 20 — FINAL TEST PERFORMANCE
# ============================================================

mae = mean_absolute_error(
    y_actual_test,
    y_pred_test
)

rmse = np.sqrt(
    mean_squared_error(
        y_actual_test,
        y_pred_test
    )
)

r2 = r2_score(
    y_actual_test,
    y_pred_test
)

print("=" * 60)
print("FINAL TEST PERFORMANCE")
print("=" * 60)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

FINAL TEST PERFORMANCE
MAE  : 7.0178
RMSE : 20.7851
R²   : 0.4628


In [28]:
# ============================================================
# CELL 21 — TRAIN / VALIDATION / TEST PERFORMANCE
# ============================================================

def calculate_metrics(
    actual,
    predicted,
    name
):
    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    r2 = r2_score(actual, predicted)

    print(f"\n{name}")
    print("-" * 40)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return mae, rmse, r2


# ------------------------------------------------------------
# TRAIN PREDICTIONS
# ------------------------------------------------------------

train_pred_scaled = model.predict(
    [
        X_seq_train_scaled,
        X_static_train_scaled
    ],
    verbose=0
).flatten()


# ------------------------------------------------------------
# VALIDATION PREDICTIONS
# ------------------------------------------------------------

val_pred_scaled = model.predict(
    [
        X_seq_val_scaled,
        X_static_val_scaled
    ],
    verbose=0
).flatten()


# ------------------------------------------------------------
# INVERSE TRANSFORMATION FUNCTION
# ------------------------------------------------------------

def convert_to_original_scale(pred_scaled):

    pred_transformed = target_scaler.inverse_transform(
        pred_scaled.reshape(-1, 1)
    ).flatten()

    return inverse_signed_log1p(
        pred_transformed
    )


# Convert predictions
train_pred = convert_to_original_scale(
    train_pred_scaled
)

val_pred = convert_to_original_scale(
    val_pred_scaled
)

# Actual values are already in original scale
train_actual = y_train
val_actual = y_val
test_actual = y_test


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

train_metrics = calculate_metrics(
    train_actual,
    train_pred,
    "TRAIN"
)

val_metrics = calculate_metrics(
    val_actual,
    val_pred,
    "VALIDATION"
)

test_metrics = calculate_metrics(
    y_actual_test,
    y_pred_test,
    "TEST"
)


TRAIN
----------------------------------------
MAE  : 30.3076
RMSE : 316.7537
R²   : 0.4841

VALIDATION
----------------------------------------
MAE  : 7.8818
RMSE : 33.0254
R²   : 0.1678

TEST
----------------------------------------
MAE  : 7.0178
RMSE : 20.7851
R²   : 0.4628


In [29]:
# ============================================================
# CELL 22 — GENERALIZATION GAP
# ============================================================

train_mae, train_rmse, train_r2 = train_metrics
val_mae, val_rmse, val_r2 = val_metrics
test_mae, test_rmse, test_r2 = test_metrics

print("=" * 60)
print("GENERALIZATION GAP")
print("=" * 60)

print(
    f"\nMAE gap  (Test - Train): "
    f"{test_mae - train_mae:.4f}"
)

print(
    f"RMSE gap (Test - Train): "
    f"{test_rmse - train_rmse:.4f}"
)

print(
    f"R² gap   (Train - Test): "
    f"{train_r2 - test_r2:.4f}"
)

GENERALIZATION GAP

MAE gap  (Test - Train): -23.2898
RMSE gap (Test - Train): -295.9685
R² gap   (Train - Test): 0.0213


In [30]:
import sys

print("Python:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python:
C:\Users\Aarjav\anaconda3\envs\inflation_project\python.exe

Python version:
3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]


In [31]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [3]:
import tslearn

print("tslearn:", tslearn.__version__)

tslearn: 0.9.0


In [4]:
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.metrics import cdist_dtw, dtw

print("All tslearn imports successful!")

All tslearn imports successful!


In [1]:
print("Kernel is alive")

import matplotlib
print("Matplotlib version:", matplotlib.__version__)

Kernel is alive
Matplotlib version: 3.11.1


In [2]:
print("Testing history...")

print("History available:", "history" in globals())

if "history" in globals():
    print("History keys:", list(history.history.keys()))
    print("Epochs:", len(history.history["loss"]))

Testing history...
History available: False


In [4]:
# ============================================================
# CELL 23A — REBUILD TEST PREDICTIONS AFTER KERNEL RESTART
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

# Load saved model
model = tf.keras.models.load_model(
    "best_gru_attention_model.keras"
)

print("Model loaded.")
print("Model inputs:")

for inp in model.inputs:
    print(inp.shape)

Model loaded.
Model inputs:
(None, 10, 1)
(None, 83)


In [5]:
# ============================================================
# CELL 23B — CHECK REQUIRED VARIABLES
# ============================================================

required = [
    "X_seq_test_scaled",
    "X_static_test_scaled",
    "y_test_scaled",
    "target_scaler"
]

print("=" * 60)
print("CHECKING TEST VARIABLES")
print("=" * 60)

for var in required:
    if var in globals():
        print("✓", var, "available")
    else:
        print("✗", var, "MISSING")

CHECKING TEST VARIABLES
✗ X_seq_test_scaled MISSING
✗ X_static_test_scaled MISSING
✗ y_test_scaled MISSING
✗ target_scaler MISSING


In [6]:
# ============================================================
# CELL 23E — CHECK SAVED PREPROCESSING FILE
# ============================================================

import pandas as pd
import numpy as np

feature_df = pd.read_csv("inflation_feature_engineered.csv")
cluster_df = pd.read_csv("final_country_clusters.csv")

print("=" * 60)
print("SAVED DATASETS")
print("=" * 60)

print("Feature dataset:", feature_df.shape)
print("Cluster dataset:", cluster_df.shape)

print("\nFeature columns:", len(feature_df.columns))
print("Cluster columns:", cluster_df.columns.tolist())

print("\nFeature dataset columns:")
print(feature_df.columns.tolist())

SAVED DATASETS
Feature dataset: (193, 151)
Cluster dataset: (193, 3)

Feature columns: 151
Cluster columns: ['Country Name', 'Country Code', 'Cluster']

Feature dataset columns:
['Country Name', 'Country Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Mean_Inflation', 'Median_Inflation', 'Std_Inflation', 'Min_Inflation', 'Max_Inflation', 'Inflation_Range', 'Mean_Recent_5Y', 'Mean_Recent_10Y', 'Mean_Previous_10Y', 'Long_Term_Change', 'Long_Term_Trend', 'Inflation_Volatility', 'High_Inflation_Years', 'Very_Hi

In [7]:
# ============================================================
# CELL 23F — RECREATE MODELING DATASET
# ============================================================

modeling_df = feature_df.merge(
    cluster_df[["Country Code", "Cluster"]],
    on="Country Code",
    how="inner"
)

print("=" * 60)
print("MODELING DATASET")
print("=" * 60)

print("Shape:", modeling_df.shape)
print("Cluster present:", "Cluster" in modeling_df.columns)

print("\nCluster counts:")
print(modeling_df["Cluster"].value_counts())

MODELING DATASET
Shape: (193, 152)
Cluster present: True

Cluster counts:
Cluster
1    126
0     67
Name: count, dtype: int64


In [8]:
# ============================================================
# CELL 23G — RECREATE THE 83 STATIC FEATURES
# ============================================================

YEAR_COLUMNS = [str(year) for year in range(1960, 2026)]

exclude_columns = (
    ["Country Name", "Country Code"]
    + YEAR_COLUMNS
    + ["Cluster"]
)

static_columns = [
    col for col in modeling_df.columns
    if col not in exclude_columns
]

static_features = modeling_df[static_columns].copy()

print("=" * 60)
print("STATIC FEATURES RECREATED")
print("=" * 60)

print("Number of static features:", len(static_columns))
print("Static feature shape:", static_features.shape)
print("Missing values:", static_features.isna().sum().sum())

STATIC FEATURES RECREATED
Number of static features: 83
Static feature shape: (193, 83)
Missing values: 0


In [9]:
# ============================================================
# CELL 23H — RECREATE 10-YEAR SEQUENCES
# ============================================================

inflation_matrix = modeling_df[YEAR_COLUMNS].astype(float).values

country_codes = modeling_df["Country Code"].values
country_names = modeling_df["Country Name"].values
clusters = modeling_df["Cluster"].values

WINDOW_SIZE = 10

X_sequences = []
X_static = []
y_values = []
sequence_clusters = []
target_years = []

for i in range(len(country_codes)):

    country_series = inflation_matrix[i]

    for t in range(WINDOW_SIZE, len(YEAR_COLUMNS)):

        X_sequences.append(
            country_series[t-WINDOW_SIZE:t]
        )

        X_static.append(
            static_features.iloc[i].values
        )

        y_values.append(
            country_series[t]
        )

        sequence_clusters.append(
            clusters[i]
        )

        target_years.append(
            int(YEAR_COLUMNS[t])
        )

X_sequences = np.array(X_sequences).reshape(-1, WINDOW_SIZE, 1)
X_static = np.array(X_static)
y_values = np.array(y_values)
sequence_clusters = np.array(sequence_clusters)
target_years = np.array(target_years)

print("=" * 60)
print("SEQUENCES RECREATED")
print("=" * 60)

print("X_sequences:", X_sequences.shape)
print("X_static   :", X_static.shape)
print("y_values   :", y_values.shape)

print("\nTarget years:")
print(target_years.min(), "-", target_years.max())

print("\nMissing values:")
print("Sequences:", np.isnan(X_sequences).sum())
print("Static:", np.isnan(X_static).sum())
print("Target:", np.isnan(y_values).sum())

SEQUENCES RECREATED
X_sequences: (10808, 10, 1)
X_static   : (10808, 83)
y_values   : (10808,)

Target years:
1970 - 2025

Missing values:
Sequences: 0
Static: 0
Target: 0


In [10]:
# ============================================================
# CELL 23I — RECREATE TRAIN / VALIDATION / TEST SPLIT
# ============================================================

TRAIN_END_YEAR = 2019
VALIDATION_END_YEAR = 2022

train_mask = target_years <= TRAIN_END_YEAR

validation_mask = (
    (target_years > TRAIN_END_YEAR) &
    (target_years <= VALIDATION_END_YEAR)
)

test_mask = target_years > VALIDATION_END_YEAR


# Sequence data
X_seq_train = X_sequences[train_mask]
X_seq_val = X_sequences[validation_mask]
X_seq_test = X_sequences[test_mask]


# Static data
X_static_train = X_static[train_mask]
X_static_val = X_static[validation_mask]
X_static_test = X_static[test_mask]


# Targets
y_train = y_values[train_mask]
y_val = y_values[validation_mask]
y_test = y_values[test_mask]


print("=" * 60)
print("TRAIN / VALIDATION / TEST SPLIT")
print("=" * 60)

print("\nTRAIN")
print(X_seq_train.shape, X_static_train.shape, y_train.shape)

print("\nVALIDATION")
print(X_seq_val.shape, X_static_val.shape, y_val.shape)

print("\nTEST")
print(X_seq_test.shape, X_static_test.shape, y_test.shape)

print("\nYears:")
print(
    "Train:",
    target_years[train_mask].min(),
    "-",
    target_years[train_mask].max()
)

print(
    "Validation:",
    target_years[validation_mask].min(),
    "-",
    target_years[validation_mask].max()
)

print(
    "Test:",
    target_years[test_mask].min(),
    "-",
    target_years[test_mask].max()
)

TRAIN / VALIDATION / TEST SPLIT

TRAIN
(9650, 10, 1) (9650, 83) (9650,)

VALIDATION
(579, 10, 1) (579, 83) (579,)

TEST
(579, 10, 1) (579, 83) (579,)

Years:
Train: 1970 - 2019
Validation: 2020 - 2022
Test: 2023 - 2025


In [11]:
# ============================================================
# CELL 23J — RECREATE SCALERS
# ============================================================

from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# Sequence scaler
# ------------------------------------------------------------

sequence_scaler = StandardScaler()

train_flat = X_seq_train.reshape(-1, 1)

sequence_scaler.fit(train_flat)

X_seq_train_scaled = sequence_scaler.transform(
    X_seq_train.reshape(-1, 1)
).reshape(X_seq_train.shape)

X_seq_val_scaled = sequence_scaler.transform(
    X_seq_val.reshape(-1, 1)
).reshape(X_seq_val.shape)

X_seq_test_scaled = sequence_scaler.transform(
    X_seq_test.reshape(-1, 1)
).reshape(X_seq_test.shape)


# ------------------------------------------------------------
# Static feature scaler
# ------------------------------------------------------------

static_scaler = StandardScaler()

static_scaler.fit(X_static_train)

X_static_train_scaled = static_scaler.transform(
    X_static_train
)

X_static_val_scaled = static_scaler.transform(
    X_static_val
)

X_static_test_scaled = static_scaler.transform(
    X_static_test
)


print("=" * 60)
print("FEATURE SCALING COMPLETED")
print("=" * 60)

print("\nSequence:")
print("Train:", X_seq_train_scaled.shape)
print("Validation:", X_seq_val_scaled.shape)
print("Test:", X_seq_test_scaled.shape)

print("\nStatic:")
print("Train:", X_static_train_scaled.shape)
print("Validation:", X_static_val_scaled.shape)
print("Test:", X_static_test_scaled.shape)

FEATURE SCALING COMPLETED

Sequence:
Train: (9650, 10, 1)
Validation: (579, 10, 1)
Test: (579, 10, 1)

Static:
Train: (9650, 83)
Validation: (579, 83)
Test: (579, 83)


In [12]:
# ============================================================
# CELL 23K — RECREATE TARGET TRANSFORMATION
# ============================================================

from sklearn.preprocessing import StandardScaler
import numpy as np

# ------------------------------------------------------------
# Signed log transformation
# ------------------------------------------------------------

def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


def inverse_signed_log1p(x):
    return np.sign(x) * np.expm1(np.abs(x))


# ------------------------------------------------------------
# Transform targets
# ------------------------------------------------------------

y_train_transformed = signed_log1p(y_train)
y_val_transformed = signed_log1p(y_val)
y_test_transformed = signed_log1p(y_test)


# ------------------------------------------------------------
# Standardize using TRAINING data only
# ------------------------------------------------------------

target_scaler = StandardScaler()

y_train_scaled = target_scaler.fit_transform(
    y_train_transformed.reshape(-1, 1)
).flatten()

y_val_scaled = target_scaler.transform(
    y_val_transformed.reshape(-1, 1)
).flatten()

y_test_scaled = target_scaler.transform(
    y_test_transformed.reshape(-1, 1)
).flatten()


print("=" * 60)
print("TARGET TRANSFORMATION RECREATED")
print("=" * 60)

print("\nTraining transformed range:")
print(
    y_train_transformed.min(),
    "to",
    y_train_transformed.max()
)

print("\nScaled target:")
print("Mean:", y_train_scaled.mean())
print("Std :", y_train_scaled.std())

print("\nMissing values:")
print("Train:", np.isnan(y_train_scaled).sum())
print("Validation:", np.isnan(y_val_scaled).sum())
print("Test:", np.isnan(y_test_scaled).sum())

TARGET TRANSFORMATION RECREATED

Training transformed range:
-2.9253325607173575 to 10.076353368307306

Scaled target:
Mean: -1.133923122352906e-16
Std : 0.9999999999999999

Missing values:
Train: 0
Validation: 0
Test: 0


In [13]:
# ============================================================
# CELL 23L — LOAD SAVED MODEL
# ============================================================

import tensorflow as tf

model = tf.keras.models.load_model(
    "best_gru_attention_model.keras"
)

print("=" * 60)
print("MODEL LOADED")
print("=" * 60)

print("Model name:", model.name)

print("\nExpected inputs:")
for inp in model.inputs:
    print(inp.name, inp.shape)

MODEL LOADED
Model name: Shared_GRU_MultiHead_Temporal_Attention

Expected inputs:
sequence_input (None, 10, 1)
static_input (None, 83)


In [14]:
# ============================================================
# CELL 23M — GENERATE TEST PREDICTIONS
# ============================================================

print("=" * 60)
print("GENERATING TEST PREDICTIONS")
print("=" * 60)

y_pred_test_scaled = model.predict(
    [
        X_seq_test_scaled,
        X_static_test_scaled
    ],
    verbose=1
).flatten()

print("\nScaled prediction shape:", y_pred_test_scaled.shape)
print("Scaled actual shape:", y_test_scaled.shape)

GENERATING TEST PREDICTIONS
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step

Scaled prediction shape: (579,)
Scaled actual shape: (579,)


In [15]:
# ============================================================
# CELL 23N — INVERSE TRANSFORM PREDICTIONS
# ============================================================

# Convert standardized prediction back
# to signed-log transformed scale

y_pred_transformed = target_scaler.inverse_transform(
    y_pred_test_scaled.reshape(-1, 1)
).flatten()

# Convert back to original inflation scale

y_pred_test = inverse_signed_log1p(
    y_pred_transformed
)

y_actual_test = y_test.copy()

print("=" * 60)
print("PREDICTIONS CONVERTED TO ORIGINAL SCALE")
print("=" * 60)

print("Actual shape    :", y_actual_test.shape)
print("Prediction shape:", y_pred_test.shape)

print("\nActual range:")
print(
    y_actual_test.min(),
    "to",
    y_actual_test.max()
)

print("\nPredicted range:")
print(
    y_pred_test.min(),
    "to",
    y_pred_test.max()
)

PREDICTIONS CONVERTED TO ORIGINAL SCALE
Actual shape    : (579,)
Prediction shape: (579,)

Actual range:
-6.60118564073726 to 254.948534781816

Predicted range:
0.06414961 to 204.53197


In [16]:
# ============================================================
# CELL 23O — FINAL TEST METRICS
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    y_actual_test,
    y_pred_test
)

rmse = np.sqrt(
    mean_squared_error(
        y_actual_test,
        y_pred_test
    )
)

r2 = r2_score(
    y_actual_test,
    y_pred_test
)

print("=" * 60)
print("FINAL TEST PERFORMANCE")
print("=" * 60)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

FINAL TEST PERFORMANCE
MAE  : 7.0178
RMSE : 20.7851
R²   : 0.4628


In [17]:
# ============================================================
# MODEL EVALUATION — R² + CROSS METRICS + PREDICTING POWER
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score
)

# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def evaluate_split(name, X_seq, X_static, y_original):
    
    # Predict
    pred_scaled = model.predict(
        [X_seq, X_static],
        verbose=0
    ).flatten()

    # Convert prediction back to original inflation scale
    pred_transformed = target_scaler.inverse_transform(
        pred_scaled.reshape(-1, 1)
    ).flatten()

    pred_original = inverse_signed_log1p(
        pred_transformed
    )

    # Metrics
    mae = mean_absolute_error(
        y_original,
        pred_original
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_original,
            pred_original
        )
    )

    r2 = r2_score(
        y_original,
        pred_original
    )

    explained_variance = explained_variance_score(
        y_original,
        pred_original
    )

    # Mean Absolute Percentage Error
    # Avoid division by zero
    non_zero = np.abs(y_original) > 1e-8

    if non_zero.sum() > 0:
        mape = np.mean(
            np.abs(
                (y_original[non_zero] - pred_original[non_zero])
                / y_original[non_zero]
            )
        ) * 100
    else:
        mape = np.nan

    return {
        "Dataset": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Explained_Variance": explained_variance,
        "MAPE_percent": mape
    }


# ------------------------------------------------------------
# Evaluate all three periods
# ------------------------------------------------------------

results = []

results.append(
    evaluate_split(
        "TRAIN",
        X_seq_train_scaled,
        X_static_train_scaled,
        y_train
    )
)

results.append(
    evaluate_split(
        "VALIDATION",
        X_seq_val_scaled,
        X_static_val_scaled,
        y_val
    )
)

results.append(
    evaluate_split(
        "TEST",
        X_seq_test_scaled,
        X_static_test_scaled,
        y_test
    )
)

metrics_df = pd.DataFrame(results)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("MODEL CROSS-METRIC EVALUATION")
print("=" * 70)

display(
    metrics_df.round(4)
)

MODEL CROSS-METRIC EVALUATION


,Dataset,MAE,RMSE,R2,Explained_Variance,MAPE_percent
0,TRAIN,30.3076,316.7537,0.4841,0.4844,37860.5679
1,VALIDATION,7.8818,33.0254,0.1678,0.1789,507.8195
2,TEST,7.0178,20.7851,0.4628,0.4714,206.0099


In [18]:
# ============================================================
# PREDICTING POWER
# ============================================================

test_r2 = metrics_df.loc[
    metrics_df["Dataset"] == "TEST",
    "R2"
].iloc[0]

test_ev = metrics_df.loc[
    metrics_df["Dataset"] == "TEST",
    "Explained_Variance"
].iloc[0]

print("=" * 70)
print("MODEL PREDICTING POWER")
print("=" * 70)

print(f"Test R²                  : {test_r2:.4f}")
print(f"Test R² percentage       : {test_r2 * 100:.2f}%")

print(
    f"Explained variance      : {test_ev:.4f}"
)

print(
    f"Explained variance (%)  : {test_ev * 100:.2f}%"
)

print(
    f"Test MAE                : "
    f"{metrics_df.loc[metrics_df['Dataset']=='TEST', 'MAE'].iloc[0]:.4f}"
)

print(
    f"Test RMSE               : "
    f"{metrics_df.loc[metrics_df['Dataset']=='TEST', 'RMSE'].iloc[0]:.4f}"
)

MODEL PREDICTING POWER
Test R²                  : 0.4628
Test R² percentage       : 46.28%
Explained variance      : 0.4714
Explained variance (%)  : 47.14%
Test MAE                : 7.0178
Test RMSE               : 20.7851
